In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import trainval_transforms, revert_normalization, revert_standardization
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir, transform=trainval_transforms)

In [3]:
from src.model import Model
from torch.utils.data import DataLoader

trainval_dl = DataLoader(dataset, 8, True)
X_batch, y_batch = next(iter(trainval_dl))
model = Model()

model.eval()
with torch.no_grad():
    preds_batch = model(X_batch)
    
preds_batch.shape, preds_batch

(torch.Size([8, 7, 7, 25]),
 tensor([[[[ 1.4908e-02, -2.1711e-02, -4.9869e-03,  ...,  1.9248e-03,
             1.6272e-03, -5.0915e-03],
           [-8.9714e-03, -5.7336e-03, -4.6616e-03,  ..., -1.3387e-02,
             5.6196e-03,  2.7690e-03],
           [-1.3451e-02, -1.0149e-03, -1.2144e-02,  ..., -5.8374e-03,
            -1.5127e-02,  2.5015e-03],
           ...,
           [ 1.6392e-02,  7.8036e-04,  1.3158e-02,  ...,  4.7763e-03,
            -1.7330e-02, -1.8557e-03],
           [-1.3968e-03,  3.1826e-03, -1.8072e-02,  ..., -1.0559e-02,
            -1.2355e-02,  3.3585e-03],
           [-1.5681e-02, -2.0619e-02,  6.9800e-03,  ..., -1.1911e-02,
             1.8296e-02, -1.9260e-02]],
 
          [[-2.5343e-02,  3.2202e-03, -7.9548e-04,  ...,  4.5930e-03,
             1.2335e-02,  6.1712e-03],
           [-1.7548e-02,  1.2365e-03,  5.6777e-03,  ..., -3.7925e-04,
            -4.2777e-03,  1.2163e-02],
           [ 9.2157e-03,  6.4438e-03, -1.7338e-03,  ..., -1.1991e-02,
           

In [4]:
from src.postprocessing import decode_preds, filter_and_group_preds, sort_class_preds_by_confidence, non_maximum_suppression

final_preds_batch = []
batch_size = preds_batch.shape[0]

for i in range(batch_size):
    preds = preds_batch[i]

    decoded_preds = decode_preds(preds)
    filtered_grouped_preds = filter_and_group_preds(decoded_preds)
    sort_class_preds_by_confidence(filtered_grouped_preds)
    suppressed_preds = non_maximum_suppression(filtered_grouped_preds)

    final_preds_batch.append(suppressed_preds)

final_preds_batch

(tensor(4.0217e-05), tensor(33.6064), tensor(-0.8742), tensor(30.6078), tensor(0.3845))
(tensor(0.0002), tensor(30.1502), tensor(97.6820), tensor(33.7947), tensor(95.2732))
(tensor(4.9892e-05), tensor(64.5614), tensor(1.6767), tensor(63.2539), tensor(-1.7118))
0
(tensor(4.9892e-05), tensor(64.5614), tensor(1.6767), tensor(63.2539), tensor(-1.7118))
(tensor(0.0002), tensor(0.6087), tensor(98.0442), tensor(-0.1883), tensor(95.0469))
(tensor(2.1225e-05), tensor(161.3979), tensor(1.5236), tensor(159.0326), tensor(-1.2441))
0
(tensor(0.0002), tensor(99.5755), tensor(95.6434), tensor(92.7385), tensor(96.2157))
0
(tensor(0.0002), tensor(99.5755), tensor(95.6434), tensor(92.7385), tensor(96.2157))
(tensor(2.1225e-05), tensor(161.3979), tensor(1.5236), tensor(159.0326), tensor(-1.2441))
0
(tensor(2.1225e-05), tensor(161.3979), tensor(1.5236), tensor(159.0326), tensor(-1.2441))
(tensor(0.0001), tensor(-0.5495), tensor(30.5286), tensor(0.4793), tensor(33.2915))
(tensor(0.0002), tensor(31.6725), t

[{'tvmonitor': [(tensor(4.0217e-05),
    tensor(33.6064),
    tensor(-0.8742),
    tensor(30.6078),
    tensor(0.3845))],
  'sofa': [(tensor(0.0002),
    tensor(30.1502),
    tensor(97.6820),
    tensor(33.7947),
    tensor(95.2732)),
   (tensor(4.9892e-05),
    tensor(64.5614),
    tensor(1.6767),
    tensor(63.2539),
    tensor(-1.7118))],
  'person': [(tensor(0.0002),
    tensor(0.6087),
    tensor(98.0442),
    tensor(-0.1883),
    tensor(95.0469)),
   (tensor(0.0002),
    tensor(99.5755),
    tensor(95.6434),
    tensor(92.7385),
    tensor(96.2157)),
   (tensor(2.1225e-05),
    tensor(161.3979),
    tensor(1.5236),
    tensor(159.0326),
    tensor(-1.2441))],
  'chair': [(tensor(0.0001),
    tensor(-0.5495),
    tensor(30.5286),
    tensor(0.4793),
    tensor(33.2915))],
  'horse': [(tensor(0.0002),
    tensor(31.6725),
    tensor(31.9107),
    tensor(31.5876),
    tensor(30.9525))],
  'aeroplane': [(tensor(6.7238e-05),
    tensor(63.9913),
    tensor(96.4269),
    tensor(62.8651

In [5]:
filtered_grouped_preds = {"chair": [torch.tensor((0.0002, 124, 111, 140, 131)), 
                                    torch.tensor((0.0003, 108, 115.5, 132, 178.5)), 
                                    torch.tensor((0.0009, 2, 146, 30, 224)),
                                    torch.tensor((0.0089, 74.5, 157.5, 113.5, 222.5)), 
                                    torch.tensor((0.0099, 117.5, 126, 144.5, 202))]}
filtered_grouped_preds

{'chair': [tensor([2.0000e-04, 1.2400e+02, 1.1100e+02, 1.4000e+02, 1.3100e+02]),
  tensor([3.0000e-04, 1.0800e+02, 1.1550e+02, 1.3200e+02, 1.7850e+02]),
  tensor([9.0000e-04, 2.0000e+00, 1.4600e+02, 3.0000e+01, 2.2400e+02]),
  tensor([8.9000e-03, 7.4500e+01, 1.5750e+02, 1.1350e+02, 2.2250e+02]),
  tensor([9.9000e-03, 1.1750e+02, 1.2600e+02, 1.4450e+02, 2.0200e+02])]}

In [6]:
suppressed_preds = non_maximum_suppression(filtered_grouped_preds)
suppressed_preds

tensor([9.9000e-03, 1.1750e+02, 1.2600e+02, 1.4450e+02, 2.0200e+02])
tensor([2.0000e-04, 1.2400e+02, 1.1100e+02, 1.4000e+02, 1.3100e+02])
tensor(0.0349)
tensor([3.0000e-04, 1.0800e+02, 1.1550e+02, 1.3200e+02, 1.7850e+02])
tensor(0.2716)
tensor([9.0000e-04, 2.0000e+00, 1.4600e+02, 3.0000e+01, 2.2400e+02])
tensor(0.)
tensor([8.9000e-03, 7.4500e+01, 1.5750e+02, 1.1350e+02, 2.2250e+02])
tensor(0.)
tensor([8.9000e-03, 7.4500e+01, 1.5750e+02, 1.1350e+02, 2.2250e+02])
tensor([2.0000e-04, 1.2400e+02, 1.1100e+02, 1.4000e+02, 1.3100e+02])
tensor(0.)
tensor([9.0000e-04, 2.0000e+00, 1.4600e+02, 3.0000e+01, 2.2400e+02])
tensor(0.)
tensor([9.0000e-04, 2.0000e+00, 1.4600e+02, 3.0000e+01, 2.2400e+02])
tensor([2.0000e-04, 1.2400e+02, 1.1100e+02, 1.4000e+02, 1.3100e+02])
tensor(0.)
tensor([2.0000e-04, 1.2400e+02, 1.1100e+02, 1.4000e+02, 1.3100e+02])


{'chair': [tensor([9.9000e-03, 1.1750e+02, 1.2600e+02, 1.4450e+02, 2.0200e+02]),
  tensor([8.9000e-03, 7.4500e+01, 1.5750e+02, 1.1350e+02, 2.2250e+02]),
  tensor([9.0000e-04, 2.0000e+00, 1.4600e+02, 3.0000e+01, 2.2400e+02]),
  tensor([2.0000e-04, 1.2400e+02, 1.1100e+02, 1.4000e+02, 1.3100e+02])]}